In [1]:
import pandas as pd
import requests
import time
import os
from dotenv import load_dotenv



In [2]:

# ==========================================
# 1. CONFIGURATION
# ==========================================
API_KEY = os.getenv("OPENWEATHER_API_KEY")
INPUT_FILE = "booking_final_gps.csv"
OUTPUT_FILE = "weather_final.csv"



In [3]:

# ==========================================
# 2. PRÉPARATION DES VILLES
# ==========================================
print(" Chargement des coordonnées...")
try:
    df_hotels = pd.read_csv(INPUT_FILE)
    df_cities = df_hotels.groupby('ville')[['latitude', 'longitude']].mean().reset_index()
    df_cities['city_id'] = df_cities.index + 1
    print(f" {len(df_cities)} villes à traiter.")
except FileNotFoundError:
    print(f" Fichier '{INPUT_FILE}' introuvable.")
    exit()


 Chargement des coordonnées...
 35 villes à traiter.


In [4]:

# ==========================================
# 3. RÉCUPÉRATION MÉTÉO (API GRATUITE 5 JOURS)
# ==========================================
print("\n Récupération des prévisions (API Standard)...")

weather_data = []



 Récupération des prévisions (API Standard)...


In [5]:

for index, row in df_cities.iterrows():
    lat = row['latitude']
    lon = row['longitude']
    city = row['ville']
    
    # URL de l'API Standard (Forecast 5 days / 3 hours) #l'api 3.0 est désormais payante, je possedais déjà une clef en 2.5
    url = f"https://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&units=metric&appid={API_KEY}"
    
    try:
        response = requests.get(url)
        data = response.json()
        
        if response.status_code == 200:
            # L'API renvoie 40 blocs de 3h (5 jours x 8 blocs)
            # On va agréger tout ça pour avoir une moyenne globale sur les 5 jours
            
            temp_sum = 0
            rain_sum = 0
            pop_sum = 0
            count = 0
            
            for item in data['list']:
                temp_sum += item['main']['temp']
                pop_sum += item.get('pop', 0)
                
                # La pluie est parfois dans 'rain' -> '3h'
                if 'rain' in item and '3h' in item['rain']:
                    rain_sum += item['rain']['3h']
                
                count += 1
            
            # Moyennes
            avg_temp = temp_sum / count
            total_rain = rain_sum # C'est un cumul
            avg_pop = (pop_sum / count) * 100
            
            # Score Météo (Arbitraire : Température - Pluie)
            score = avg_temp - (total_rain * 2)
            
            weather_data.append({
                'city_id': row['city_id'],
                'ville': city,
                'latitude': lat,
                'longitude': lon,
                'temperature_moyenne': round(avg_temp, 1),
                'pluie_totale_mm': round(total_rain, 1),
                'risque_pluie_pct': round(avg_pop, 0),
                'score_meteo': round(score, 1)
            })
            print(f"    {city} : {round(avg_temp, 1)}°C, Pluie: {round(total_rain, 1)}mm")
            
        else:
            print(f"    Erreur {response.status_code} pour {city}: {data.get('message')}")
            
    except Exception as e:
        print(f"    Crash sur {city} : {e}")
        
    time.sleep(0.2) # Petite pause


    Aigues Mortes : 14.2°C, Pluie: 8.9mm
    Aix en Provence : 15.0°C, Pluie: 2.9mm
    Amiens : 9.1°C, Pluie: 6.4mm
    Annecy : 8.8°C, Pluie: 20.3mm
    Ariege : 5.3°C, Pluie: 15.4mm
    Avignon : 12.7°C, Pluie: 5.0mm
    Bayeux : 9.9°C, Pluie: 5.0mm
    Bayonne : 10.6°C, Pluie: 29.1mm
    Besancon : 7.9°C, Pluie: 43.0mm
    Biarritz : 10.9°C, Pluie: 27.3mm
    Bormes les Mimosas : 14.6°C, Pluie: 0.6mm
    Carcassonne : 10.7°C, Pluie: 6.5mm
    Cassis : 15.0°C, Pluie: 2.6mm
    Chateau du Haut Koenigsbourg : 6.7°C, Pluie: 33.4mm
    Collioure : 14.4°C, Pluie: 6.5mm
    Colmar : 9.3°C, Pluie: 37.5mm
    Dijon : 8.3°C, Pluie: 36.4mm
    Eguisheim : 9.0°C, Pluie: 39.4mm
    Gorges du Verdon : 11.1°C, Pluie: 10.1mm
    Grenoble : 9.6°C, Pluie: 30.2mm
    La Rochelle : 11.6°C, Pluie: 2.5mm
    Le Havre : 10.3°C, Pluie: 2.0mm
    Lille : 10.0°C, Pluie: 5.3mm
    Lyon : 9.1°C, Pluie: 36.1mm
    Marseille : 15.4°C, Pluie: 3.0mm
    Mont Saint Michel : 9.8°C, Pluie: 4.6mm
    Montauban : 9.8°

In [6]:

# ==========================================
# 4. CLASSEMENT ET SAUVEGARDE
# ==========================================
if weather_data:
    df_weather = pd.DataFrame(weather_data)
    
    # Tri par "Meilleur temps" (Score le plus élevé)
    df_weather = df_weather.sort_values(by='score_meteo', ascending=False)
    
    # Sauvegarde
    df_weather.to_csv(OUTPUT_FILE, index=False)
    
    
    print("\n" + "="*40)
    print(" TOP 5 DESTINATIONS (MÉTÉO)")
    print("="*40)
    print(df_weather[['ville', 'temperature_moyenne', 'pluie_totale_mm']].head(5).to_string(index=False))
    
    
    print(f"\n Fichier '{OUTPUT_FILE}' généré avec succès.")
else:
    print("\n Aucune donnée récupérée.")


 TOP 5 DESTINATIONS (MÉTÉO)
             ville  temperature_moyenne  pluie_totale_mm
Bormes les Mimosas                 14.6              0.6
            Cassis                 15.0              2.6
         Marseille                 15.4              3.0
   Aix en Provence                 15.0              2.9
       La Rochelle                 11.6              2.5

 Fichier 'weather_final.csv' généré avec succès.
